## Building Approximate GP Models from Scratch

When working with large datasets, exact GP models can become computationally expensive due to their O(n³) complexity. Approximate GP models provide a practical alternative by using sparse approximations or reduced-rank methods to scale to larger datasets while maintaining good predictive performance.

In this example, we demonstrate how to build and compare approximate GP models with analytical (exact) GP models using the 3D Rosenbrock function. We'll examine the trade-offs between model construction time and data size, showing how approximate models enable efficient modeling for larger datasets.

In [ ]:
# set values if testing
import os

from xopt import Xopt, Evaluator
from xopt.generators import RandomGenerator
from xopt.resources.test_functions.rosenbrock import (
    evaluate_rosenbrock,
    make_rosenbrock_vocs,
)

from xopt.generators.bayesian.models.approximate import ApproximateModelConstructor
from xopt.generators.bayesian.models.standard import StandardModelConstructor

# Ignore all warnings
import warnings

warnings.filterwarnings("ignore")


SMOKE_TEST = os.environ.get("SMOKE_TEST")
MAX_TRAINING_DATA = 100 if SMOKE_TEST else 2500
MAX_STEPS = 1 if SMOKE_TEST else 10

# make rosenbrock function vocs in 3D
vocs = make_rosenbrock_vocs(3)

# collect some data using random sampling
evaluator = Evaluator(function=evaluate_rosenbrock)
generator = RandomGenerator(vocs=vocs)
X = Xopt(generator=generator, evaluator=evaluator)
X.random_evaluate(MAX_TRAINING_DATA)

## Benchmark: Standard vs Approximate Model Construction

To understand the practical benefits of approximate models, we'll benchmark the construction time of both approaches across varying dataset sizes. This comparison demonstrates how approximate models provide computational advantages while maintaining model quality. Emperical test runs show that Approximate GP models should outperform Standard models around 2000 points. 

### Notes
 - This benchmark can be computationally expensive to evaluate, so it may take a long time to execute and you may run out of memory.
 - This benchmark measures the amount of time needed to fit the model hyperparameters to the dataset, which includes both the execution time of the model and the number of steps needed to optimize the hyperparameters. As a result, the training time will not show perfect O(n³) scaling, but should show rough qualatative agreement.

In [ ]:
data = X.data

In [ ]:
model_constructor = StandardModelConstructor()
approximate_model_constructor = ApproximateModelConstructor()

In [ ]:
import time
import numpy as np

import matplotlib.pyplot as plt
from scipy.optimize import curve_fit


data_sizes = np.linspace(50, MAX_TRAINING_DATA, MAX_STEPS, dtype=int)
standard_times = []
approximate_times = []

standard_model_constructor = StandardModelConstructor()
approximate_model_constructor = ApproximateModelConstructor()

for n in data_sizes:
    subset = data.iloc[:n]

    # Time StandardModelConstructor
    start = time.time()
    standard_model_constructor.build_model_from_vocs(vocs, subset)
    standard_times.append(time.time() - start)

    # Time ApproximateModelConstructor
    start = time.time()
    approximate_model_constructor.build_model_from_vocs(vocs, subset)
    approximate_times.append(time.time() - start)

    print(
        f"n={n}: Standard={standard_times[-1]:.3f}s, Approximate={approximate_times[-1]:.3f}s"
    )


# fit n^3 scaling to standard times
def cubic(x, a):
    return a * x**3


# Fit cubic curve to standard times
params, _ = curve_fit(cubic, data_sizes, standard_times)
fitted_standard_times = cubic(data_sizes, *params)

# Plot results
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(data_sizes, standard_times, marker="o", label="StandardModelConstructor")
ax.plot(data_sizes, approximate_times, marker="s", label="ApproximateModelConstructor")
ax.plot(data_sizes, fitted_standard_times, linestyle="--", label="Cubic Fit (Standard)")

ax.set_xlabel("Number of data points")
ax.set_ylabel("Build time (s)")
ax.set_title("GP Model Construction Time Benchmark")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()